# CopyWrite AI - run on a free GPU

**Every time you open this notebook:**
1. **Runtime > Change runtime type > T4 GPU > Save** (only needed once per notebook).
2. **Runtime > Run all** (or press `Ctrl+F9`).
3. Wait about 3-5 minutes. Step 4 prints two links. **Click Link 1** (Link 2 also works on your phone) - that is the app.
4. In the app: pick a sample handwriting (or upload yours), paste your notes, click **Write my notes**.

The first click on *Write my notes* can take 1-2 extra minutes while the models finish loading.
A page of notes takes roughly 1-3 minutes on a T4 with Quality "Fast", longer with "Best".

**If something looks wrong:** open *More options*, tick **Save debug images**, generate again, then download the
zip and copy the *Run details* text. The zip shows every piece the model wrote and whether it was accepted or rewritten.

The link works for about 72 h, but the app stops when the Colab session ends (idle ~90 min, or a few hours at most).
To get the newest code after a `git push`: **Runtime > Restart session and run all**. Cell 2 pulls the latest version.

In [ ]:
# 1. Check the GPU
import subprocess
try:
    print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]).decode())
except Exception:
    print("NO GPU FOUND. Runtime > Change runtime type > T4 GPU > Save, then Runtime > Run all again.")

In [ ]:
# 2. Get the code (clones the first time, pulls updates afterwards)
import os
REPO_URL = "https://github.com/NishaadG/CopyWrite-AI.git"
os.chdir("/content")
if os.path.isdir("copywrite_repo/.git"):
    !git -C copywrite_repo pull -q
else:
    !git clone -q $REPO_URL copywrite_repo
%cd /content/copywrite_repo
!git log -1 --format="Using commit %h: %s"

In [ ]:
# 3. Install (about 1-2 minutes)
!pip install -q -r requirements.txt -r requirements-model.txt

### 4. Launch the app
The app runs in the background. This cell prints two links when it is ready (about 1 minute):
- **Link 1** goes through Colab itself. It is the most reliable and works in this browser while you are signed in.
- **Link 2** (`gradio.live`) is public and works on any device, e.g. your phone, but its relay sometimes times out (504).

Run the cell again to restart the app.

In [ ]:
# 4. Launch the app in the background and print the links
import re, subprocess, sys, time
from google.colab.output import eval_js

if "app_proc" in globals() and app_proc.poll() is None:  # restart if already running
    app_proc.terminate(); app_proc.wait()
app_proc = subprocess.Popen([sys.executable, "-u", "app.py", "--share", "--port", "7860"],
                            stdout=open("/content/app.log", "w"), stderr=subprocess.STDOUT)
share = None
for _ in range(150):  # wait up to 2.5 min for the public link
    time.sleep(1)
    log = open("/content/app.log").read()
    m = re.search(r"https://[\w.-]+\.gradio\.live", log)
    if m:
        share = m.group(0); break
    if app_proc.poll() is not None:
        break
if app_proc.poll() is not None:
    print("The app stopped. Its output (copy this if you need help):\n"); print(open("/content/app.log").read())
else:
    print("Link 1 (most reliable, this browser):", eval_js("google.colab.kernel.proxyPort(7860)"))
    print("Link 2 (public, any device):        ", share or "not ready - run the next cell to see why")

**If a link shows an error, times out, or generation fails**, run this cell and copy its output.
It shows the app's log: the models loading, every generation, and any error message.

In [ ]:
!tail -n 60 /content/app.log
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv
!free -h | head -2

---
### Optional extras

**Quick model test** - writes two lines in a sample handwriting without the app, checks them with the OCR, and
saves every intermediate image to `debug_test/` (`debug_test/pieces.csv` lists what passed and what was rewritten):

In [ ]:
import time
from copywrite import EmuruGenerator
from copywrite.library import load_library
from copywrite.ocr import get_ocr
from copywrite.writer import LineWriter

ref = load_library()[0]
display(ref.image); print("style text:", ref.text)
gen = EmuruGenerator(); gen.load()
t = time.time()
writer = LineWriter(gen, [ref], ocr=get_ocr(), seed=0, debug_dir="debug_test")
lines = writer.write(["Generative AI mini project for the final year", "handwriting generation with a transformer"])
print(f"{time.time() - t:.1f}s", writer.report.as_dict())
for w in lines.values():
    display(w.image)

**Command line / evaluation** - upload your photo to `samples/style/` first (Files panel on the left):

In [ ]:
# !python -m copywrite.cli --text samples/text/sample_notes.txt --style samples/style/me.jpg --backend emuru --quality best --debug-dir outputs/run1/debug --out outputs/run1
# !pip install -q -r requirements-extra.txt
# !python scripts/evaluate.py --photos samples/style/me_12lines.jpg --text samples/style/me_12lines.txt --k 2 --backends emuru font --out outputs/eval_me